# Prerequisite, Ray Data Essentials

## TLDR

Ray Data is a streaming data library for AI workloads. It loads, transforms, and
feeds data across a cluster of CPUs and GPUs without pulling everything into
memory. This optional pre-reading walks the core workflow on an MNIST image
example, loading from S3, transforming with `map_batches`, running batch
inference with a stateful actor, and computing simple aggregates. Notebook 01
uses Ray Data to feed a distributed training job, so this is the foundation.


## Introduction

Most of this course is about Ray Train. But training is only as fast as the data
you can feed it, and foundation model datasets do not fit in memory. Ray Data
solves that with a streaming execution engine. It processes data in a pipeline
across the cluster, so while the GPU trains on one batch the CPUs prepare the
next.

You write ordinary Python. Your NumPy and PyTorch code runs as is, and Ray Data
handles the distribution, the batching, and the memory. This notebook gives you
just enough Ray Data to follow notebook 01, where a Ray Dataset feeds a four-GPU
training job.

We use a small MNIST image dataset hosted on public S3 so everything runs
quickly on this cluster.


## Key concepts used in this notebook

**Dataset and blocks.** A Ray Dataset is a distributed collection of blocks,
each a chunk of rows held in the object store. Operators process blocks in
parallel across the cluster.

**Lazy execution.** Transforms build a plan. Nothing runs until you consume the
data with a method like `take`, `count`, `write_parquet`, or `iter_batches`.

**`map_batches`.** The main transform. It applies your function to each batch in
parallel. Use a plain function for stateless work and a class with an actor pool
for stateful work like loading a model once.

**`ActorPoolStrategy`.** Runs a class-based transform on a pool of long-lived
actors, so expensive setup like loading a model happens once per actor, not once
per batch.

**Streaming.** Ray Data pipelines data through operators rather than
materializing each stage in full, which keeps memory bounded and hardware busy.


## What you will learn

- How to load data into a Ray Dataset from S3
- Why execution is lazy, and what triggers it
- How to transform data with `map_batches`
- How to run batch inference with a stateful actor that loads a model once
- How to materialize, inspect, and aggregate results
- Where Ray Data fits in the training pipeline you build in notebook 01


## Why Ray Data

| Challenge | Without Ray Data | With Ray Data |
|---|---|---|
| Dataset bigger than memory | Manual chunking and paging | Streaming execution, bounded memory |
| Feed GPUs without stalling | Data loading blocks the GPU | CPU preprocessing overlaps GPU compute |
| Preprocess across a cluster | Hand-built distributed pipeline | `map` and `map_batches` scale out |
| Load many formats and sources | A connector per format | Built-in readers for S3, Parquet, images, HuggingFace |
| Feed distributed training | Custom samplers and sharding | `get_dataset_shard` in Ray Train |


## How this scales on Anyscale

| | This notebook | Production |
|---|---|---|
| Data size | 500 MNIST images from S3 | Terabytes streamed from object storage |
| Compute | A small CPU and GPU cluster | Hundreds of CPUs feeding many GPUs |
| Memory | Fits easily | Streamed, never fully materialized |
| Change needed | none | A larger source path and more workers |


## Cell 1 — Connect and load images from S3

**What you do.** Connect to Ray and read a folder of MNIST images straight from
public S3 into a Ray Dataset.

**What to check.** The dataset reports 500 rows, and each row has an `image` and
a `path`. The read is distributed across the cluster as parallel tasks.

**Why it matters.** Loading is the first stage of any pipeline. Ray Data reads
remote storage in parallel and hands you a dataset of blocks, without downloading
everything to one machine first.


In [ ]:
import os
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"

import numpy as np
import ray
from common import utils

if not ray.is_initialized():
    ray.init(address="auto", runtime_env=utils.build_runtime_env())

ds = ray.data.read_images(
    "s3://anyscale-public-materials/ray-ai-libraries/mnist/50_per_index/",
    include_paths=True,
)
print("rows:", ds.count())
ds.schema()


## Cell 2 — Lazy execution and a quick peek

**What you do.** Pull five rows with `take`. The label lives in the file path,
so we read it from there.

**What to check.** Only now does any work happen. `take` materializes just five
rows, not the whole dataset.

**Why it matters.** Lazy execution lets Ray Data plan and optimize the whole
pipeline before running it, and lets you inspect a few rows cheaply.


In [ ]:
sample = ds.take(5)
for row in sample:
    label = row["path"].split("/")[-2]
    print("label", label, "image shape", np.array(row["image"]).shape)


## Cell 3 — Transform with map_batches

**What you do.** Normalize the images with a stateless function. `map_batches`
applies it to each batch in parallel. This stays lazy.

**What to check.** The call returns immediately. The transform runs later, when a
downstream step consumes the data.

**Why it matters.** `map_batches` is the workhorse transform. For vectorized work
like array math it is faster than row-by-row `map`, because it operates on a
whole batch at once.


In [ ]:
from torchvision.transforms import Compose, ToTensor, Normalize

def normalize(batch):
    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
    batch["image"] = [transform(img) for img in batch["image"]]
    return batch

ds_normalized = ds.map_batches(normalize)
print(ds_normalized)


## Cell 4 — Batch inference with a stateful actor

**What you do.** Run an MNIST classifier over the normalized images. The model is
wrapped in a class so it loads once per actor, then scores many batches. We
download the model to shared storage first.

**What to check.** `compute=ActorPoolStrategy(size=1)` puts the transform on one
long-lived actor. The model loads in `__init__`, not on every batch.

**Why it matters.** This is the pattern for batch inference and for any transform
with expensive setup. Loading a model once per actor instead of once per batch is
the difference between fast and unusable.


In [ ]:
import subprocess
MODEL_PATH = "/mnt/cluster_storage/mnist_model.pt"
subprocess.run(
    f"aws s3 cp --only-show-errors --no-sign-request "
    f"s3://anyscale-public-materials/ray-ai-libraries/mnist/model/model.pt {MODEL_PATH}",
    shell=True, check=True,
)

import torch

class MNISTClassifier:
    def __init__(self, model_path):
        self.model = torch.jit.load(model_path)
        self.model.eval()

    def __call__(self, batch):
        images = torch.tensor(np.array(batch["image"])).float()
        with torch.no_grad():
            logits = self.model(images).numpy()
        batch["predicted_label"] = np.argmax(logits, axis=1)
        return batch

ds_preds = ds_normalized.map_batches(
    MNISTClassifier,
    fn_constructor_kwargs={"model_path": MODEL_PATH},
    batch_size=100,
    num_cpus=1,
    compute=ray.data.ActorPoolStrategy(size=1),
)

for p in ds_preds.take(5):
    true_label = p["path"].split("/")[-2]
    print("true", true_label, "predicted", p["predicted_label"])


## Cell 5 — Materialize and inspect

**What you do.** Execute the whole pipeline once and cache the result in the
object store with `materialize`. Then read the execution stats.

**What to check.** After materialize, the actor pool is released and the stats
show per-operator timing and throughput.

**Why it matters.** Materialize is the right tool when you will reuse a result
many times or when a downstream operation needs the full dataset. For a
one-pass streaming pipeline you would skip it.


In [ ]:
ds_preds = ds_preds.materialize()
print(ds_preds.stats())


## Cell 6 — Aggregate with groupby

**What you do.** Add the true label from the path, then compute per-label
accuracy with `groupby` and `map_groups`.

**What to check.** You get one accuracy number per digit, computed in parallel
across groups.

**Why it matters.** Beyond row transforms, Ray Data does grouped and aggregate
operations, so you can summarize results at scale without leaving the dataset.


In [ ]:
def add_label(row):
    row["ground_truth_label"] = int(row["path"].split("/")[-2])
    return row

def accuracy(group):
    return {
        "ground_truth_label": group["ground_truth_label"][:1],
        "accuracy": [float(np.mean(group["predicted_label"] == group["ground_truth_label"]))],
    }

acc_by_label = (
    ds_preds.map(add_label)
    .groupby("ground_truth_label")
    .map_groups(accuracy)
    .sort("ground_truth_label")
    .to_pandas()
)
acc_by_label


## Going further

Ray Data has more than fits in a prerequisite. A few directions to explore when
your workloads grow.

- **Shuffling.** Three strategies trade randomness for cost, from a cheap
  file-level shuffle, to block-order shuffle, to a full global `random_shuffle`.
- **Performance tuning.** Set `num_cpus`, `num_gpus`, and block size per operator.
  Ray Data applies backpressure so a fast stage does not overwhelm a slow one.
- **Fault tolerance.** Failed tasks retry automatically, and RayTurbo Data adds
  job-level checkpointing that skips already-processed rows on restart.
- **Preprocessors and expressions.** Built-in scikit-learn style preprocessors
  and a column expression API cover common feature work without custom code.

The point for this course is the workflow you just ran. Load, transform, and
stream. In notebook 01 you hand a Ray Dataset to Ray Train and each worker pulls
its own shard.


## Conclusion

You loaded images from S3 into a Ray Dataset, transformed them lazily with
`map_batches`, ran batch inference with a stateful actor that loaded the model
once, materialized the result, and computed per-label accuracy with a grouped
aggregation.

Ray Data ideas you used. Datasets and blocks, lazy execution, `map` and
`map_batches`, `ActorPoolStrategy` for stateful transforms, `materialize`, and
`groupby` with `map_groups`.

Next, in notebook 01, a Ray Dataset like this one feeds a distributed training
job. Each worker calls `get_dataset_shard` and trains on its own slice while Ray
Data streams and preprocesses in the background.
